In [ ]:
import numpy as np
import pandas as pd
from nltk.classify import NaiveBayesClassifier
from nltk.corpus import subjectivity
from nltk.sentiment import SentimentAnalyzer
from nltk.sentiment.util import *

import numpy as np 
import pandas as pd 
#pd.set_option('display.max_colwidth', -1)
from time import time
import re
import string
import os
from pprint import pprint
import collections

import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="darkgrid")
sns.set(font_scale=1.3)

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.metrics import classification_report

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
#from sklearn.externals import joblib
import joblib as jb

#import gensim

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

import warnings
warnings.filterwarnings('ignore')

np.random.seed(37)

SHOW_GRAPHS = True

# VIX

In [ ]:
from yahoo_fin.stock_info import get_data

df_vix  = pd.DataFrame()
df_vix = get_data("^VIX", start_date="25/09/2019", end_date="09/10/2021", index_as_date = True, interval="1d")
df_vix.index = pd.to_datetime(df_vix.index)
df_vix["Date"]= pd.to_datetime(df_vix.index)

df_vix["vix"] = df_vix["adjclose"]
df_vix.drop(columns=["open", "high", "low", "close", "adjclose", "ticker", "volume"], inplace=True)
df_vix["vix"]  = df_vix["vix"].ffill()

df_vix["vix_yesterday"] = df_vix["vix"].shift(periods=1)
# vix RETURN
df_vix["vix_return"]=(df_vix["vix"]-df_vix["vix_yesterday"])/df_vix["vix_yesterday"]
# vix LN RETURN
df_vix["vix_ln_return"]=np.log(df_vix["vix"]/df_vix["vix_yesterday"])
# vix EWM
df_vix["vix_ewm"]=df_vix["vix"].ewm(10).mean()

In [ ]:
df_vix

In [ ]:
df

In [ ]:
df_vix

# S&P500

In [ ]:
## S&P500
df_sp500 = get_data("^GSPC", start_date="25/09/2019", end_date="09/10/2021", index_as_date = True, interval="1d")
df_sp500 = get_data("^GSPC", start_date="25/09/2019", end_date="09/10/2021", index_as_date = True, interval="1d")
df_sp500.index = pd.to_datetime(df_sp500.index)
df_sp500["Date"]= pd.to_datetime(df_sp500.index)

df_sp500["sp500"] = df_sp500["adjclose"]
df_sp500.drop(columns=["open", "high", "low", "close", "adjclose", "ticker", "volume"], inplace=True)
df_sp500["sp500"]  = df_sp500["sp500"].ffill()

df_sp500["sp500_yesterday"] = df_sp500["sp500"].shift(periods=1)
# sp500 RETURN
df_sp500["sp500_return"]=(df_sp500["sp500"]-df_sp500["sp500_yesterday"])/df_sp500["sp500_yesterday"]
# sp500 LN RETURN
df_sp500["sp500_ln_return"]=np.log(df_sp500["sp500"]/df_sp500["sp500_yesterday"])
# sp500 EWM
df_sp500["sp500_ewm"]=df_sp500["sp500"].ewm(10).mean()
df_sp500

# SENTIMENT

In [ ]:
df_09_15=pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/Sentiment_09_15.pkl', compression=None)
df_16_08=pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/Sentiment_16_08.pkl', compression=None)
df_09_15.index= pd.to_datetime(df_09_15.index)
df_16_08.index= pd.to_datetime(df_16_08.index)
df_09_15.Date= pd.to_datetime(df_09_15.Date)
df_16_08.Date= pd.to_datetime(df_16_08.Date)

In [ ]:
df_09_15 = pd.merge(df_vix,df_09_15,how='outer', on='Date')
df_09_15 = pd.merge(df_sp500,df_09_15,how='outer', on='Date')
df_16_08 = pd.merge(df_vix,df_16_08,how='outer', on='Date')
df_16_08 = pd.merge(df_sp500,df_16_08,how='outer', on='Date')

In [ ]:
# Prepare data df_09_15
df_09_15['year'] = [d.year for d in df_09_15.Date]
df_09_15['month'] = [d.strftime('%b') for d in df_09_15.Date]
df_09_15['day'] = [d.day_name() for d in df_09_15.Date]
years = df_09_15['year'].unique()
df_09_15['month-year'] = [d.strftime('%b') + str(d.year) for d in df_09_15.Date]

# Prepare data df_16_08
df_16_08['year'] = [d.year for d in df_16_08.Date]
df_16_08['month'] = [d.strftime('%b') for d in df_16_08.Date]
df_16_08['day'] = [d.day_name() for d in df_16_08.Date]
years = df_16_08['year'].unique()
df_16_08['month-year'] = [d.strftime('%b') + str(d.year) for d in df_16_08.Date]


In [ ]:
df_09_15 = df_09_15.merge(df_vix,on='Date',how='left')
df_09_15 = df_09_15.merge(df_sp500, on='Date', how='left')

df_16_08 = df_16_08.merge(df_vix,on='Date',how='left')
df_16_08 = df_16_08.merge(df_sp500, on='Date', how='left')

In [ ]:
df_09_15.index= pd.to_datetime(df_09_15.Date)
df_16_08.index= pd.to_datetime(df_16_08.Date)
df_09_15.Date= pd.to_datetime(df_09_15.Date)
df_16_08.Date= pd.to_datetime(df_16_08.Date)

In [ ]:
df_09_15["sentiment"] = df_09_15["sentiment"].apply(np.float)
df_16_08["sentiment"] = df_16_08["sentiment"].apply(np.float)

In [ ]:
sentiment_yesterday = df_09_15["sentiment"].shift(periods=1)
sentiment_yesterday = sentiment_yesterday.ffill()
df_09_15["sentiment_ln_return"]=np.log(df_09_15["sentiment"]/sentiment_yesterday)


sentiment_yesterday = df_16_08["sentiment"].shift(periods=1)
sentiment_yesterday = sentiment_yesterday.ffill()
df_16_08["sentiment_ln_return"]=np.log(df_16_08["sentiment"]/sentiment_yesterday)

In [ ]:
df_09_15["sentiment"] = df_09_15["sentiment"].ffill()
df_16_08["sentiment"] = df_16_08["sentiment"].ffill()

In [ ]:
sentiment_yesterday = df_09_15["sentiment"].shift(periods=1)
df_09_15["sentiment_return"]=((df_09_15["sentiment"]-sentiment_yesterday)/sentiment_yesterday)
sentiment_yesterday = df_16_08["sentiment"].shift(periods=1)
df_16_08["sentiment_return"]=((df_16_08["sentiment"]-sentiment_yesterday)/sentiment_yesterday)

In [ ]:
df_09_15["sentiment_return"] = df_09_15["sentiment_return"].ffill()
df_16_08["sentiment_return"] = df_16_08["sentiment_return"].ffill()

In [ ]:
# df_part3['Num_word'] = df_part3['Articles'].apply(lambda x:len(str(x).split())) #Number Of words in main text